# B2-019-attention-transformers — Practice p07 — Solution

**Type:** constrained-coding · **Difficulty:** intro · **Concepts:** matrix-transpose, scaled-dot-product-attention

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

For each batch, the score contraction uses x @ x.swapaxes(-1,-2), giving one N by N score matrix. Stable row softmax is applied over the final key axis, then weights @ x restores the feature width D. No batch or token loop is needed.

In [ ]:
import numpy as np
SEED = 20260808
ATOL = 1e-10
RTOL = 1e-10

def batched_self_attention_np(x):
    if not isinstance(x, np.ndarray) or x.dtype != np.float64:
        raise TypeError("x must be a float64 NumPy array")
    if x.ndim != 3 or 0 in x.shape:
        raise ValueError("x must have nonempty shape (B,N,D)")
    if not np.all(np.isfinite(x)):
        raise ValueError("x must be finite")
    scores = (x @ x.swapaxes(-1, -2)) / np.sqrt(x.shape[-1])
    shifted = scores - np.max(scores, axis=-1, keepdims=True)
    numerators = np.exp(shifted)
    weights = numerators / np.sum(numerators, axis=-1, keepdims=True)
    output = weights @ x
    return scores, weights, output

x = np.arange(12, dtype=np.float64).reshape(2, 3, 2) / 10.0
scores, weights, output = batched_self_attention_np(x)
manual_score = np.dot(x[1, 2], x[1, 0]) / np.sqrt(2.0)
EXPECTED_WEIGHTS = np.array([[[0.3286305558090359, 0.33331111222217036, 0.33805833196879376], [0.3100601588542632, 0.3327784714129649, 0.35716136973277196], [0.2919174458747102, 0.33154059587912177, 0.376541958246168]], [[0.2742554965112529, 0.3296092638682006, 0.3961352396205465], [0.2571223526246465, 0.32700267701695673, 0.4158749703583968], [0.24056047653213902, 0.3237450643689399, 0.4356944590989212]]], dtype=np.float64)
EXPECTED_OUTPUT = np.array([[[0.2018855552319516, 0.3018855552319516], [0.2094202421757018, 0.30942024217570174], [0.21692490247429155, 0.3169249024742915]], [[0.8243759486218587, 0.9243759486218587], [0.8317505235467502, 0.9317505235467501], [0.8390267965133565, 0.9390267965133566]]], dtype=np.float64)

### Answer check

In [ ]:
assert scores.shape == weights.shape == (2, 3, 3)
assert output.shape == (2, 3, 2)
assert scores.dtype == weights.dtype == output.dtype == np.float64
assert np.isclose(scores[1, 2, 0], manual_score, atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(weights, EXPECTED_WEIGHTS, atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(output, EXPECTED_OUTPUT, atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(weights.sum(axis=-1), np.ones((2, 3)), atol=ATOL, rtol=RTOL)
assert np.all(np.isfinite(output))